## 12 代码解析

SessionsPythonTool是专门设计用于连接[Azure Container Apps](https://portal.azure.com/#browse/Microsoft.App%2FcontainerApps) (ACA) Dynamic Sessions的，必须用Azure账号和部署。

In [3]:
from semantic_kernel.contents import ChatMessageContent, FunctionCallContent, FunctionResultContent
from semantic_kernel.core_plugins import SessionsPythonTool

async def handle_intermediate_steps(message: ChatMessageContent) -> None:
    for item in message.items or []:
        if isinstance(item, FunctionResultContent):
            print(f"# Function Result:> {item.result}")
        elif isinstance(item, FunctionCallContent):
            print(f"# Function Call:> {item.name} with arguments: {item.arguments}")
        else:
            print(f"# {message.name}: {message} ")

In [4]:
from azure.identity import AzureCliCredential
resources_dir = "./python-samples/getting_started_with_agents/resources/"
# 命令行 az login
python_code_interpreter = SessionsPythonTool(
    credential=AzureCliCredential(),
    allowed_upload_directories=[resources_dir],
)

Failed to load the ACASessionsSettings with message: 1 validation error for ACASessionsSettings
pool_management_endpoint
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing


FunctionInitializationError: KernelFunction failed to initialize: Failed to load the ACASessionsSettings with message: 1 validation error for ACASessionsSettings
pool_management_endpoint
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [ ]:
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

agent_code = ChatCompletionAgent(
    service=OpenAIChatCompletion(),
    name="Host",
    instructions="Answer questions about the menu.",
    plugins=[python_code_interpreter],
)

In [ ]:
from pathlib import Path

csv_file_path = Path(resources_dir) / "sales.csv"
file_metadata = await python_code_interpreter.upload_file(local_file_path=csv_file_path.as_posix())

In [ ]:
TASK = (
    "What's the total sum of all sales for all segments using Python? "
    f"Use the uploaded file {file_metadata.full_path} for reference."
)
print(f"【User】'{TASK}'")

async for response in agent_code.invoke(
    messages=TASK,
    on_intermediate_message=handle_intermediate_steps,
):
    print(f"【{response.name}】{response}")